# scRareBench | High-Level Seurat (CCA / RPCA) | Dataset 2 — mBDRC renal cortex

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This notebook is the Dataset 2 Seurat workflow after the executed all-pairs RPCA
run reached the Seurat integration stage and failed internally during hierarchical
reference merging with:

`FindIntegrationMatrix -> subscript out of bounds`

The preprocessing before that point was successful:
- Dataset 2: 97,125 cells × 36,030 registered features
- 32 integration batches (`donor_id × assay`)
- zero-library cells over the full registered feature universe: 0
- all split counts/data layers: `dgCMatrix`
- ScaleData: successful
- PCA: 97,125 × 50, successful
- failure occurred only during RPCA's internal multi-dataset merge

## Reference-based correction

Seurat's `RPCAIntegration` and `CCAIntegration` both support a `reference`
argument. Importantly, Seurat's internal `PairwiseIntegrateReference` has a
special single-reference path: when exactly one reference dataset is supplied,
it does **not** hierarchically merge reference datasets. Queries are mapped to
the fixed reference instead.

This notebook therefore uses exactly **one deterministic reference batch** for
both RPCA and CCA.

### Reference selection policy

No biological cell-type labels are used. The reference is selected once from
method metadata only:

1. Build the 32 registered `donor_id × assay` batches.
2. Find donor(s) with the maximum number of distinct assay types.
3. Among those donors, choose the donor with the largest total cell count.
4. Within that donor, choose its largest batch as the single Seurat reference.
5. Use the same fixed reference for RPCA, CCA, and all method seeds.

This favors a well-supported donor with broad assay coverage while keeping the
actual Seurat reference to one layer, which removes the hierarchical
reference-merge path that failed in the executed all-pairs run.

## Dataset 2 contract retained

- selector: 2 (`mbdrc_renal_cortex`)
- evaluation label: `cell_type` — never passed to Seurat
- integration/evaluation batch: `scrarebench_batch = donor_id × assay`
- full registered Dataset 2 feature universe
- 4000 global Seurat-v3 HVGs from raw counts
- HVG span = 0.3
- no batch key during HVG selection
- official Seurat `NormalizeData(LogNormalize)`
- PCA = 50
- integration dimensions = 30 fixed
- method seeds = 42, 123, 2026
- benchmark seed = 42
- full seed-42 preflight before multi-seed benchmarking

> Use a High-RAM Colab runtime. Reference-based integration should reduce the
> problematic merge path, but CCA on ~97k cells is still CPU/RAM intensive.


## V2 correction after executed single-reference run

The executed single-reference run successfully completed all Dataset 2
preprocessing, found anchors from all 31 query layers to the fixed reference,
and then failed inside `IntegrateEmbeddings` with:

`CreateAssayObject(): No cell names (colnames) names present in the input matrix`

Root cause: the notebook passed `dims.to.integrate = ndims`, where `ndims` is
the scalar integer `30`. In Seurat, `dims.to.integrate` is a **vector of
dimension indices**, not a count. Passing `30` therefore means "integrate only
dimension 30". In the single-reference branch that creates a one-row temporary
matrix, R drops it to a vector and the cell column names are lost.

The successful Dataset 0 notebook does not set `dims.to.integrate` at all; it
only sets `dims = 1:30`. Seurat's own tests likewise use vectors such as
`dims.to.integrate = 1:10`, never a scalar count.

V2 therefore removes `dims.to.integrate` from **both RPCA and CCA**, exactly
matching Dataset 0. Seurat then keeps the dimensions of the original PCA
reduction, and this notebook explicitly validates that at least 30 integrated
dimensions exist before saving the first 30 as the benchmark representation.


## V3 correction after executed V2 run

The executed V2 notebook confirmed that the previous `dims.to.integrate`
correction worked: single-reference RPCA progressed into real query-by-query
integration. The new failure occurred during query 5:

`Number of anchor cells is less than k.weight. Consider lowering k.weight to less than 84`

Seurat defaults to `k.weight = 100`. For this reference/query pair, anchor
finding produced 148 anchor pairs but only 84 unique query-side anchor cells
after deduplication, so 100 weighting neighbors are impossible.

V3 uses one fixed, reproducible `k.weight = 50` for **both RPCA and CCA and all
three method seeds**. This directly follows Seurat's recommended remedy, stays
well below the observed 84-cell limit, and avoids any seed-dependent/adaptive
fallback. The value is included in the cache key, method-policy provenance,
and R runtime log.

## 1. Install scRareBench v0.10.6


In [ ]:
import subprocess
import sys

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    "scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-deps",
    SCRAREBENCH_GITHUB,
])

from scrarebench.runtime import (
    print_install_report,
    setup_runtime,
)

runtime_report = setup_runtime(
    extra_requirements=(
        "scikit-misc>=0.3",
    ),
    extra_imports=(
        "skmisc",
    ),
    quiet=False,
)
print_install_report(runtime_report)

import scrarebench

if scrarebench.__version__ != "0.10.6":
    raise RuntimeError(
        "This notebook requires scRareBench 0.10.6, "
        f"but imported {scrarebench.__version__}."
    )

print("scRareBench version:", scrarebench.__version__)


## 2. Install and validate Seurat v5


In [ ]:
import os
import shutil
import subprocess

env = os.environ.copy()
env["DEBIAN_FRONTEND"] = "noninteractive"

subprocess.check_call(
    ["apt-get", "update", "-qq"],
    env=env,
)

subprocess.check_call([
    "apt-get",
    "install",
    "-y",
    "-qq",
    "r-base",
    "build-essential",
    "gfortran",
    "libcurl4-openssl-dev",
    "libssl-dev",
    "libxml2-dev",
    "libhdf5-dev",
    "libfontconfig1-dev",
    "libfreetype6-dev",
    "libpng-dev",
    "libtiff-dev",
    "libjpeg-dev",
    "libharfbuzz-dev",
    "libfribidi-dev",
    "libgit2-dev",
    "libglpk-dev",
], env=env)

if shutil.which("Rscript") is None:
    raise RuntimeError(
        "Rscript was not installed successfully."
    )

r_install = r'''
options(
  repos = c(CRAN = "https://cloud.r-project.org"),
  timeout = 1800
)

need_seurat <- (
  !requireNamespace("Seurat", quietly = TRUE) ||
  packageVersion("Seurat") < package_version("5.0.0")
)

if (need_seurat) {
  install.packages(
    "Seurat",
    dependencies = NA,
    Ncpus = 2
  )
}

if (!requireNamespace("Seurat", quietly = TRUE)) {
  stop("Seurat installation failed.")
}

if (
  packageVersion("Seurat") <
  package_version("5.0.0")
) {
  stop("Seurat >= 5.0.0 is required.")
}

suppressPackageStartupMessages(library(Seurat))

required_functions <- c(
  "IntegrateLayers",
  "RPCAIntegration",
  "CCAIntegration",
  "NormalizeData",
  "ScaleData",
  "RunPCA"
)

missing_functions <- required_functions[
  !vapply(
    required_functions,
    exists,
    logical(1),
    mode = "function"
  )
]

if (length(missing_functions) > 0) {
  stop(
    paste(
      "Missing required Seurat functions:",
      paste(missing_functions, collapse = ", ")
    )
  )
}

integrate_args <- names(formals(IntegrateLayers))
if (!"features" %in% integrate_args) {
  stop(
    "Installed Seurat IntegrateLayers() does not expose a features argument."
  )
}

for (fun_name in c("RPCAIntegration", "CCAIntegration")) {
  fun_args <- names(formals(get(fun_name)))
  required_args <- c(
    "features",
    "dims",
    "dims.to.integrate",
    "normalization.method"
  )
  missing_args <- setdiff(required_args, fun_args)
  if (length(missing_args) > 0) {
    stop(
      paste0(
        fun_name,
        " is missing required arguments: ",
        paste(missing_args, collapse = ", ")
      )
    )
  }
}

cat(
  "Seurat version:",
  as.character(packageVersion("Seurat")),
  "\n"
)
cat(
  "SeuratObject version:",
  as.character(packageVersion("SeuratObject")),
  "\n"
)
cat("Seurat API preflight: OK\n")
'''

subprocess.check_call([
    "Rscript",
    "-e",
    r_install,
])


## 3. Load Dataset 2 and recover validated raw counts


In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

from scrarebench import (
    dataset_info,
    load_dataset,
)

adata = load_dataset(2)
info = dataset_info(adata)

print(adata)
print("\nDataset metadata:")
for key, value in info.items():
    print(f"  {key}: {value}")

if info.get("dataset_index") != 2:
    raise RuntimeError(
        f"Expected Dataset 2; observed: {info.get('dataset_index')}"
    )

if info.get("dataset_key") != "mbdrc_renal_cortex":
    raise RuntimeError(
        f"Unexpected Dataset 2 key: {info.get('dataset_key')!r}"
    )

if info.get("label_key") != "cell_type":
    raise RuntimeError(
        f"Unexpected Dataset 2 label key: {info.get('label_key')!r}"
    )

batch_key = info.get("batch_key")
if not batch_key or batch_key not in adata.obs.columns:
    raise KeyError(
        f"Registered Dataset 2 batch key is invalid: {batch_key!r}"
    )

if info.get("scib_hvg_batch_mode") != "global":
    raise RuntimeError(
        "Dataset 2 must use the registered global HVG policy."
    )

if not adata.obs_names.is_unique:
    raise ValueError(
        "Dataset 2 observation names must be unique."
    )

def sampled_count_diagnostics(
    matrix,
    max_values=100_000,
    seed=42,
):
    values = (
        matrix.data
        if sp.issparse(matrix)
        else np.asarray(matrix).ravel()
    )
    values = np.asarray(values)

    if len(values) > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[
            rng.choice(
                len(values),
                size=max_values,
                replace=False,
            )
        ]

    values = values.astype(float, copy=False)

    if len(values) == 0:
        return {
            "finite": True,
            "nonnegative": True,
            "integer_like": True,
        }

    return {
        "finite": bool(np.isfinite(values).all()),
        "nonnegative": bool(values.min() >= 0),
        "integer_like": bool(
            np.allclose(
                values,
                np.rint(values),
                rtol=0.0,
                atol=1e-6,
            )
        ),
    }

def copy_matrix(matrix):
    if sp.issparse(matrix):
        return matrix.copy().tocsr()
    return np.asarray(matrix).copy()

def is_valid_counts(matrix):
    diag = sampled_count_diagnostics(matrix)
    return (
        diag["finite"]
        and diag["nonnegative"]
        and diag["integer_like"]
    ), diag

def ensure_counts_layer(
    adata,
    layer_key="counts",
):
    # 1) Prefer an existing valid count layer.
    if layer_key in adata.layers:
        valid, diag = is_valid_counts(
            adata.layers[layer_key]
        )
        if valid:
            return f"layers/{layer_key}", diag
        print(
            f"WARNING: existing {layer_key!r} layer is not raw-count-like; "
            "trying adata.raw.X instead."
        )

    # 2) Recover aligned raw counts from adata.raw.X.
    if adata.raw is not None:
        raw_names = pd.Index(
            adata.raw.var_names.astype(str)
        )
        current_names = pd.Index(
            adata.var_names.astype(str)
        )

        if not raw_names.is_unique:
            raise ValueError(
                "adata.raw.var_names are not unique; "
                "cannot safely align raw counts."
            )

        positions = raw_names.get_indexer(
            current_names
        )

        if np.all(positions >= 0):
            raw_aligned = adata.raw.X[:, positions]
            valid, diag = is_valid_counts(
                raw_aligned
            )
            if valid:
                adata.layers[layer_key] = (
                    copy_matrix(raw_aligned)
                )
                return (
                    "raw.X aligned to adata.var_names",
                    diag,
                )

    # 3) Last resort: X only if it is genuinely raw-count-like.
    valid, diag = is_valid_counts(adata.X)
    if valid:
        adata.layers[layer_key] = copy_matrix(
            adata.X
        )
        return "adata.X", diag

    raise ValueError(
        "Could not recover nonnegative integer-like raw counts "
        "for Dataset 2."
    )

COUNT_LAYER = (
    info.get("count_layer")
    or "counts"
)

count_source, count_diag = (
    ensure_counts_layer(
        adata,
        COUNT_LAYER,
    )
)

if COUNT_LAYER not in adata.layers:
    raise RuntimeError(
        "Validated count layer was not attached to adata."
    )

print("\nValidated Dataset 2 contract")
print("  cells:", f"{adata.n_obs:,}")
print("  features:", f"{adata.n_vars:,}")
print("  label:", info["label_key"])
print("  batch:", info["batch_key"])
print(
    "  batches:",
    adata.obs[info["batch_key"]].nunique(),
)
print("  count layer:", COUNT_LAYER)
print("  count source:", count_source)
print("  count diagnostics:", count_diag)

print("\nFeature metadata diagnostics (informational only):")
if "feature_biotype" in adata.var.columns:
    print(
        "  feature_biotype:",
        adata.var["feature_biotype"]
        .astype(str)
        .value_counts(dropna=False)
        .to_dict(),
    )
if "feature_type" in adata.var.columns:
    print(
        "  feature_type unique values:",
        int(
            adata.var["feature_type"]
            .astype(str)
            .nunique(dropna=False)
        ),
    )

batch_sizes = (
    adata.obs[info["batch_key"]]
    .astype(str)
    .value_counts()
    .sort_values()
)
print("\nBatch-size diagnostics:")
print("  min:", int(batch_sizes.min()))
print("  max:", int(batch_sizes.max()))
print("  median:", float(batch_sizes.median()))
display(batch_sizes.rename("n_cells").to_frame())


## 4. Reproducibility and method configuration


In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

BATCH_KEY = str(info["batch_key"])
BATCH_COMPONENTS = list(
    info.get(
        "batch_components",
        [],
    )
)

if BATCH_COMPONENTS != [
    "donor_id",
    "assay",
]:
    raise RuntimeError(
        "Dataset 2 reference selection expects "
        "batch_components=['donor_id', 'assay']; "
        f"observed {BATCH_COMPONENTS!r}."
    )

for key in (
    BATCH_KEY,
    "donor_id",
    "assay",
):
    if key not in adata.obs.columns:
        raise KeyError(
            f"Dataset 2 is missing required obs column {key!r}."
        )

# ------------------------------------------------------------
# Build one row per registered integration batch.
# Each scrarebench_batch must map to exactly one donor + assay.
# ------------------------------------------------------------
batch_frame = (
    adata.obs[
        [
            BATCH_KEY,
            "donor_id",
            "assay",
        ]
    ]
    .astype(str)
    .copy()
)

batch_integrity = (
    batch_frame
    .groupby(
        BATCH_KEY,
        observed=True,
    )
    .agg(
        donor_nunique=(
            "donor_id",
            "nunique",
        ),
        assay_nunique=(
            "assay",
            "nunique",
        ),
    )
)

if (
    (batch_integrity["donor_nunique"] != 1).any()
    or
    (batch_integrity["assay_nunique"] != 1).any()
):
    raise RuntimeError(
        "At least one registered Dataset 2 batch does not map "
        "uniquely to one donor_id and one assay."
    )

batch_table = (
    batch_frame
    .groupby(
        BATCH_KEY,
        observed=True,
    )
    .agg(
        donor_id=(
            "donor_id",
            "first",
        ),
        assay=(
            "assay",
            "first",
        ),
        n_cells=(
            BATCH_KEY,
            "size",
        ),
    )
    .reset_index()
)

# ------------------------------------------------------------
# Reference policy:
# 1) maximum assay coverage per donor
# 2) maximum total cells among those donors
# 3) deterministic lexical donor tie-break
# 4) largest batch within the selected donor
# ------------------------------------------------------------
donor_summary = (
    batch_table
    .groupby(
        "donor_id",
        observed=True,
    )
    .agg(
        assay_count=(
            "assay",
            "nunique",
        ),
        total_cells=(
            "n_cells",
            "sum",
        ),
        batch_count=(
            BATCH_KEY,
            "size",
        ),
    )
    .reset_index()
)

max_assay_coverage = int(
    donor_summary[
        "assay_count"
    ].max()
)

reference_donor_row = (
    donor_summary[
        donor_summary[
            "assay_count"
        ]
        == max_assay_coverage
    ]
    .sort_values(
        [
            "total_cells",
            "donor_id",
        ],
        ascending=[
            False,
            True,
        ],
        kind="mergesort",
    )
    .iloc[0]
)

REFERENCE_DONOR = str(
    reference_donor_row[
        "donor_id"
    ]
)

reference_batch_row = (
    batch_table[
        batch_table[
            "donor_id"
        ]
        == REFERENCE_DONOR
    ]
    .sort_values(
        [
            "n_cells",
            BATCH_KEY,
        ],
        ascending=[
            False,
            True,
        ],
        kind="mergesort",
    )
    .iloc[0]
)

REFERENCE_BATCH = str(
    reference_batch_row[
        BATCH_KEY
    ]
)
REFERENCE_ASSAY = str(
    reference_batch_row[
        "assay"
    ]
)
REFERENCE_N_CELLS = int(
    reference_batch_row[
        "n_cells"
    ]
)

REFERENCE_POLICY = (
    "max_assay_coverage_then_max_donor_cells_"
    "then_largest_batch"
)

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_policy": (
        "global_seurat_v3_raw_counts"
    ),
    "hvg_batch_key": None,
    "hvg_span": 0.3,
    "normalize_scale_factor": 10000.0,
    "npcs": 50,
    "integration_dims": 30,
    "k_weight": 50,
    "reference_policy": (
        REFERENCE_POLICY
    ),
    "reference_batch": (
        REFERENCE_BATCH
    ),
    "reference_donor": (
        REFERENCE_DONOR
    ),
    "reference_assay": (
        REFERENCE_ASSAY
    ),
    "reference_n_cells": (
        REFERENCE_N_CELLS
    ),
}

OUTPUT_ROOT = (
    "/content/"
    "scrarebench_seurat_dataset2_reference_based"
)
SEURAT_ROOT = (
    "/content/"
    "scrarebench_seurat_dataset2_reference_runtime"
)

batch_sizes = (
    adata.obs[
        BATCH_KEY
    ]
    .astype(str)
    .value_counts()
    .sort_values()
)

if int(
    batch_sizes.min()
) <= int(
    METHOD_CONFIG[
        "integration_dims"
    ]
):
    raise RuntimeError(
        "The fixed Seurat recipe requires every integration batch "
        f"to contain more than {METHOD_CONFIG['integration_dims']} "
        "cells."
    )

if (
    REFERENCE_N_CELLS
    <= METHOD_CONFIG[
        "integration_dims"
    ]
):
    raise RuntimeError(
        "Selected reference is too small for the fixed "
        "integration dimensionality."
    )

print("Method seeds:", METHOD_SEEDS)
print("Benchmark seed:", BENCHMARK_SEED)
print("Integration batches:", len(batch_table))
print(
    "Assays:",
    sorted(
        batch_table[
            "assay"
        ].unique()
    ),
)
print(
    "Maximum assay coverage per donor:",
    max_assay_coverage,
)
print(
    "Selected reference donor:",
    REFERENCE_DONOR,
)
print(
    "Selected reference assay:",
    REFERENCE_ASSAY,
)
print(
    "Selected reference batch:",
    REFERENCE_BATCH,
)
print(
    "Selected reference cells:",
    REFERENCE_N_CELLS,
)
print(
    "Reference policy:",
    REFERENCE_POLICY,
)
print(
    "Fixed Seurat k.weight:",
    METHOD_CONFIG["k_weight"],
)

display(
    donor_summary
    .sort_values(
        [
            "assay_count",
            "total_cells",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(10)
)

display(
    batch_table[
        batch_table[
            "donor_id"
        ]
        == REFERENCE_DONOR
    ]
    .sort_values(
        "n_cells",
        ascending=False,
    )
)


## 5. User-owned Seurat runner

The runner performs these steps for each method seed:

1. Recover only GEX/RNA features if an explicit feature-type column exists.
2. Select exactly **4000 global Seurat-v3 HVGs** in Python from raw counts.
3. Export the **full raw GEX count matrix** to Seurat.
4. Export the 4000 HVGs by positional index, avoiding duplicate-name mismatch.
5. Split the Seurat RNA assay by the registered Dataset 2 batch.
6. Run `LogNormalize` on full GEX counts.
7. Set the fixed 4000 genes as `VariableFeatures`.
8. Run `ScaleData` and 50-PC PCA on those 4000 genes.
9. Run RPCA and CCA from the same base object, independently.
10. Force each integrated representation to 30 dimensions.
11. Validate/reorder output barcodes back to exact Dataset 2 order.

No benchmark labels are supplied to the method.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import io, sparse

from scrarebench import (
    MethodOutput,
    dataset_info,
)

SEURAT_CACHE = {}
SEURAT_ROOT = Path(SEURAT_ROOT)
SEURAT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R_SCRIPT = r'''
args <- commandArgs(trailingOnly = TRUE)

workdir <- args[[1]]
seed <- as.integer(args[[2]])
npcs <- as.integer(args[[3]])
ndims <- as.integer(args[[4]])
scale_factor <- as.numeric(args[[5]])
reference_batch <- as.character(args[[6]])
k_weight <- as.integer(args[[7]])

if (k_weight < 1) {
  stop("k.weight must be >= 1")
}

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
})

if (packageVersion("Seurat") < package_version("5.0.0")) {
  stop("Seurat >= 5.0.0 is required.")
}
if (ndims > npcs) {
  stop("integration_dims must be <= npcs")
}

set.seed(seed)
options(future.globals.maxSize = 40 * 1024^3)

counts <- as(
  Matrix::readMM(file.path(workdir, "counts.mtx")),
  "dgCMatrix"
)
genes <- read.csv(file.path(workdir, "genes.csv"), stringsAsFactors = FALSE)
barcodes <- read.csv(file.path(workdir, "barcodes.csv"), stringsAsFactors = FALSE)
meta <- read.csv(
  file.path(workdir, "metadata.csv"),
  stringsAsFactors = FALSE,
  check.names = FALSE
)
hvg_indices <- read.csv(
  file.path(workdir, "hvg_indices.csv"),
  stringsAsFactors = FALSE
)

if (nrow(counts) != nrow(genes)) stop("counts/gene dimension mismatch")
if (ncol(counts) != nrow(barcodes)) stop("counts/barcode dimension mismatch")
if (nrow(meta) != nrow(barcodes)) stop("metadata/barcode dimension mismatch")
if (!"BATCH" %in% colnames(meta)) stop("metadata.csv is missing BATCH")
if (!"index_1based" %in% colnames(hvg_indices)) {
  stop("hvg_indices.csv is missing index_1based")
}

rownames(counts) <- make.unique(as.character(genes$gene))
colnames(counts) <- as.character(barcodes$barcode)

if (anyDuplicated(colnames(counts)) > 0) stop("Input barcodes are not unique")

if (
  any(hvg_indices$index_1based < 1) ||
  any(hvg_indices$index_1based > nrow(counts))
) {
  stop("HVG indices are outside the feature universe")
}

rownames(meta) <- as.character(meta$barcode)
meta <- meta[colnames(counts), , drop = FALSE]

if (!all(rownames(meta) == colnames(counts))) {
  stop("Metadata/barcode alignment failed")
}

meta$BATCH <- factor(meta$BATCH)

if (nlevels(meta$BATCH) < 2) {
  stop("Seurat integration requires at least two batches")
}

batch_sizes <- table(meta$BATCH)
cat(
  "R integration batches: n=", length(batch_sizes),
  ", min=", min(batch_sizes),
  ", max=", max(batch_sizes),
  "\n",
  sep = ""
)

if (min(batch_sizes) <= ndims) {
  stop(
    paste0(
      "Smallest batch has ", min(batch_sizes),
      " cells, but fixed integration dimensionality is ", ndims, "."
    )
  )
}

library_sizes <- Matrix::colSums(counts)
zero_library_cells <- sum(library_sizes <= 0)

cat(
  "Zero-library cells in full registered feature universe: ",
  zero_library_cells,
  "\n",
  sep = ""
)

# The executed full-feature Dataset 2 run has zero zero-library cells.
# Stop rather than silently changing the benchmark recipe if that ever changes.
if (zero_library_cells > 0) {
  stop(
    paste0(
      "Found ",
      zero_library_cells,
      " zero-library cells in the full registered Dataset 2 feature universe. ",
      "This notebook intentionally stops instead of altering cells/counts."
    )
  )
}

obj <- CreateSeuratObject(
  counts = counts,
  meta.data = meta,
  assay = "RNA",
  project = "scRareBench_Dataset2_Seurat",
  min.cells = 0,
  min.features = 0
)

if (ncol(obj) != nrow(barcodes)) {
  stop("CreateSeuratObject changed the benchmark cell count")
}
if (nrow(obj) != nrow(counts)) {
  stop("CreateSeuratObject changed the feature count")
}
if (!all(rownames(obj) == rownames(counts))) {
  stop(
    "CreateSeuratObject changed feature order; positional HVG mapping is unsafe"
  )
}

selected_features <- rownames(obj)[
  hvg_indices$index_1based
]

if (length(selected_features) != 4000) {
  stop(
    paste0(
      "Expected 4000 fixed Dataset 2 HVGs, got ",
      length(selected_features)
    )
  )
}
if (anyDuplicated(selected_features) > 0) {
  stop("Selected HVGs are not unique inside Seurat")
}

# Match the successful Dataset 0 Seurat workflow:
# split counts first, then let Seurat create normalized data layers.
obj[["RNA"]] <- split(
  obj[["RNA"]],
  f = obj$BATCH
)

count_layers <- Layers(
  obj[["RNA"]],
  search = "counts"
)

if (length(count_layers) != nlevels(meta$BATCH)) {
  stop(
    paste0(
      "Expected ",
      nlevels(meta$BATCH),
      " count layers, got ",
      length(count_layers)
    )
  )
}

count_layer_classes <- vapply(
  count_layers,
  function(layer_name) {
    class(
      LayerData(
        obj[["RNA"]],
        layer = layer_name
      )
    )[1]
  },
  character(1)
)

cat(
  "Count-layer classes: ",
  paste(unique(count_layer_classes), collapse = ", "),
  "\n",
  sep = ""
)

if (any(count_layer_classes != "dgCMatrix")) {
  stop(
    paste0(
      "All split count layers must be dgCMatrix before NormalizeData; observed: ",
      paste(unique(count_layer_classes), collapse = ", ")
    )
  )
}

obj <- NormalizeData(
  obj,
  normalization.method = "LogNormalize",
  scale.factor = scale_factor,
  verbose = FALSE
)

data_layers <- Layers(
  obj[["RNA"]],
  search = "data"
)

if (length(data_layers) != nlevels(meta$BATCH)) {
  stop(
    paste0(
      "Expected ",
      nlevels(meta$BATCH),
      " normalized data layers, got ",
      length(data_layers)
    )
  )
}

data_layer_classes <- vapply(
  data_layers,
  function(layer_name) {
    class(
      LayerData(
        obj[["RNA"]],
        layer = layer_name
      )
    )[1]
  },
  character(1)
)

cat(
  "Data-layer classes after NormalizeData: ",
  paste(unique(data_layer_classes), collapse = ", "),
  "\n",
  sep = ""
)

if (any(data_layer_classes != "dgCMatrix")) {
  stop(
    paste0(
      "All normalized data layers must be dgCMatrix for Seurat StitchMatrix; observed: ",
      paste(unique(data_layer_classes), collapse = ", ")
    )
  )
}

data_layer_sizes <- vapply(
  data_layers,
  function(layer_name) {
    ncol(
      LayerData(
        obj[["RNA"]],
        layer = layer_name
      )
    )
  },
  integer(1)
)

if (min(data_layer_sizes) <= ndims) {
  stop(
    paste0(
      "A normalized data layer has only ",
      min(data_layer_sizes),
      " cells, insufficient for fixed ",
      ndims,
      " dimensions."
    )
  )
}

cat(
  "Split + NormalizeData validated: ",
  length(data_layers),
  " batches; smallest layer=",
  min(data_layer_sizes),
  "\n",
  sep = ""
)

# --------------------------------------------------------------
# Resolve exactly one Seurat reference layer.
# Use cell membership rather than layer-name string parsing.
# --------------------------------------------------------------
layer_batches <- vapply(
  data_layers,
  function(layer_name) {
    layer_cells <- Cells(
      obj[["RNA"]],
      layer = layer_name
    )

    values <- unique(
      as.character(
        obj$BATCH[
          layer_cells
        ]
      )
    )

    if (length(values) != 1) {
      stop(
        paste0(
          "Data layer ",
          layer_name,
          " maps to ",
          length(values),
          " BATCH values; expected exactly one."
        )
      )
    }

    values[[1]]
  },
  character(1)
)

if (anyDuplicated(layer_batches) > 0) {
  stop(
    "Multiple Seurat data layers map to the same registered BATCH."
  )
}

reference_idx <- which(
  layer_batches
  == reference_batch
)

if (length(reference_idx) != 1) {
  stop(
    paste0(
      "Could not resolve exactly one reference layer for BATCH='",
      reference_batch,
      "'. Matches=",
      length(reference_idx)
    )
  )
}

reference_layer <- data_layers[
  reference_idx
]

reference_cells <- Cells(
  obj[["RNA"]],
  layer = reference_layer
)

cat(
  "Reference-based integration:\n",
  "  BATCH = ", reference_batch, "\n",
  "  layer = ", reference_layer, "\n",
  "  layer index = ", reference_idx, "\n",
  "  cells = ", length(reference_cells), "\n",
  "  fixed k.weight = ", k_weight, "\n",
  sep = ""
)

if (length(reference_cells) <= ndims) {
  stop(
    paste0(
      "Resolved reference layer has only ",
      length(reference_cells),
      " cells; fixed integration dimensions are ",
      ndims,
      "."
    )
  )
}

writeLines(
  c(
    paste0("reference_batch=", reference_batch),
    paste0("reference_layer=", reference_layer),
    paste0("reference_index=", reference_idx),
    paste0("reference_n_cells=", length(reference_cells))
  ),
  file.path(
    workdir,
    "seurat_reference.txt"
  )
)

VariableFeatures(obj) <- selected_features

obj <- ScaleData(
  obj,
  features = selected_features,
  verbose = FALSE
)

scaled_features <- Features(
  obj,
  assay = "RNA",
  layer = "scale.data"
)

missing_scaled <- setdiff(
  selected_features,
  scaled_features
)

if (length(missing_scaled) > 0) {
  stop(
    paste0(
      length(missing_scaled),
      " fixed HVGs are missing from scale.data"
    )
  )
}

obj <- RunPCA(
  obj,
  features = selected_features,
  npcs = npcs,
  seed.use = seed,
  verbose = FALSE
)

pca_emb <- Embeddings(obj[["pca"]])

if (nrow(pca_emb) != ncol(obj)) stop("PCA cell count mismatch")
if (ncol(pca_emb) < ndims) stop("PCA produced fewer dimensions than required")
if (any(!is.finite(pca_emb))) stop("PCA contains NaN/Inf")

cat(
  "Base PCA: ",
  nrow(pca_emb),
  " x ",
  ncol(pca_emb),
  "\n",
  sep = ""
)

rm(pca_emb)
gc()

save_latent <- function(object, reduction_name, output_name) {
  if (!reduction_name %in% Reductions(object)) {
    stop(paste("Missing reduction:", reduction_name))
  }

  emb <- Embeddings(object[[reduction_name]])

  if (nrow(emb) != ncol(object)) {
    stop(paste(reduction_name, "cell count mismatch"))
  }
  if (ncol(emb) < ndims) {
    stop(paste(reduction_name, "has fewer than", ndims, "dimensions"))
  }

  emb <- emb[, seq_len(ndims), drop = FALSE]

  if (any(!is.finite(emb))) {
    stop(paste(reduction_name, "contains NaN/Inf"))
  }

  out <- data.frame(
    barcode = rownames(emb),
    emb,
    check.names = FALSE
  )

  write.csv(
    out,
    file.path(workdir, output_name),
    row.names = FALSE,
    quote = TRUE
  )
}

cat("Starting RPCAIntegration...\n")
set.seed(seed)

obj <- IntegrateLayers(
  object = obj,
  method = RPCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.rpca",
  features = selected_features,
  layers = data_layers,
  reference = reference_idx,
  normalization.method = "LogNormalize",
  dims = seq_len(ndims),
  k.weight = k_weight,
  verbose = TRUE
)

rpca_n_dims <- ncol(
  Embeddings(
    obj[["integrated.rpca"]]
  )
)

cat(
  "RPCA integrated reduction dimensions: ",
  rpca_n_dims,
  "\n",
  sep = ""
)

if (rpca_n_dims < ndims) {
  stop(
    paste0(
      "RPCA integrated reduction has only ",
      rpca_n_dims,
      " dimensions; at least ",
      ndims,
      " are required."
    )
  )
}

save_latent(
  obj,
  "integrated.rpca",
  "seurat_rpca_latent.csv"
)

cat("RPCAIntegration completed.\n")

# Latent is saved; remove the extra reduction to lower peak memory before CCA.
obj[["integrated.rpca"]] <- NULL
gc()

cat("Starting CCAIntegration...\n")
set.seed(seed)

obj <- IntegrateLayers(
  object = obj,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",
  features = selected_features,
  layers = data_layers,
  reference = reference_idx,
  normalization.method = "LogNormalize",
  dims = seq_len(ndims),
  k.weight = k_weight,
  verbose = TRUE
)

cca_n_dims <- ncol(
  Embeddings(
    obj[["integrated.cca"]]
  )
)

cat(
  "CCA integrated reduction dimensions: ",
  cca_n_dims,
  "\n",
  sep = ""
)

if (cca_n_dims < ndims) {
  stop(
    paste0(
      "CCA integrated reduction has only ",
      cca_n_dims,
      " dimensions; at least ",
      ndims,
      " are required."
    )
  )
}

save_latent(
  obj,
  "integrated.cca",
  "seurat_cca_latent.csv"
)

cat("CCAIntegration completed.\n")
gc()

writeLines(
  selected_features,
  file.path(workdir, "seurat_selected_genes.txt")
)

writeLines(
  Layers(obj[["RNA"]]),
  file.path(workdir, "seurat_layers.txt")
)

sink(file.path(workdir, "seurat_session_info.txt"))
print(sessionInfo())
sink()

cat(
  "Seurat seed ",
  seed,
  " completed successfully.\n",
  sep = ""
)
'''

R_SYNTAX_PATH = (
    SEURAT_ROOT
    / "_seurat_dataset2_syntax_check.R"
)

R_SYNTAX_PATH.write_text(
    R_SCRIPT,
    encoding="utf-8",
)

r_parse_check = subprocess.run(
    [
        "Rscript",
        "-e",
        (
            "parse(file="
            + repr(str(R_SYNTAX_PATH))
            + "); cat('R syntax check: OK\\n')"
        ),
    ],
    text=True,
    capture_output=True,
)

if r_parse_check.stdout:
    print(r_parse_check.stdout, end="")

if r_parse_check.returncode != 0:
    raise RuntimeError(
        "Generated Seurat R script failed to parse.\n"
        "---- R stdout ----\n"
        f"{r_parse_check.stdout}\n"
        "---- R stderr ----\n"
        f"{r_parse_check.stderr}"
    )

if r_parse_check.stderr:
    print(r_parse_check.stderr, end="")


def prepare_seurat_inputs(
    method_adata,
    workdir,
    config,
):
    meta = dataset_info(method_adata)

    batch_key = str(
        meta.get("batch_key")
        or ""
    )
    count_layer = str(
        meta.get("count_layer")
        or "counts"
    )

    if (
        not batch_key
        or batch_key not in
        method_adata.obs.columns
    ):
        raise KeyError(
            "Dataset 2 batch metadata is missing."
        )

    if (
        count_layer not in
        method_adata.layers
    ):
        raise KeyError(
            f"Raw count layer {count_layer!r} is required for Seurat."
        )

    if not method_adata.obs_names.is_unique:
        raise ValueError(
            "Dataset 2 cell barcodes must be unique."
        )

    if not method_adata.var_names.is_unique:
        raise ValueError(
            "Dataset 2 feature identifiers must be unique."
        )

    # Full registered Dataset 2 feature universe.
    # No feature_type/modality regex filtering.
    feature_names = pd.Index(
        method_adata.var_names
        .astype(str)
    )

    counts = (
        method_adata.layers[
            count_layer
        ]
    )
    counts = (
        counts.tocsr()
        if sparse.issparse(counts)
        else sparse.csr_matrix(counts)
    )

    if counts.shape != (
        method_adata.n_obs,
        method_adata.n_vars,
    ):
        raise RuntimeError(
            "Unexpected Dataset 2 count-matrix shape."
        )

    if counts.nnz:
        sample_size = min(
            counts.nnz,
            100_000,
        )

        rng = np.random.default_rng(42)

        positions = rng.choice(
            counts.nnz,
            size=sample_size,
            replace=False,
        )

        sample = counts.data[
            positions
        ].astype(float, copy=False)

        if not np.isfinite(sample).all():
            raise ValueError(
                "Raw counts contain NaN/Inf."
            )
        if sample.min() < 0:
            raise ValueError(
                "Raw counts contain negative values."
            )
        if not np.allclose(
            sample,
            np.rint(sample),
            rtol=0.0,
            atol=1e-6,
        ):
            raise ValueError(
                "Raw counts are not integer-like."
            )

    hvg_adata = ad.AnnData(
        X=counts,
        obs=pd.DataFrame(
            index=method_adata.obs_names.copy()
        ),
        var=pd.DataFrame(
            index=feature_names.copy()
        ),
    )

    hvg_kwargs = {
        "flavor": "seurat_v3",
        "n_top_genes": int(
            config["n_hvg"]
        ),
        "span": float(
            config["hvg_span"]
        ),
        "subset": False,
        "check_values": True,
    }

    if (
        config.get("hvg_batch_key")
        is not None
    ):
        hvg_kwargs["batch_key"] = (
            config["hvg_batch_key"]
        )

    sc.pp.highly_variable_genes(
        hvg_adata,
        **hvg_kwargs,
    )

    hvg_mask = (
        hvg_adata.var[
            "highly_variable"
        ]
        .fillna(False)
        .to_numpy(dtype=bool)
    )

    n_selected = int(
        hvg_mask.sum()
    )

    if n_selected != int(
        config["n_hvg"]
    ):
        raise RuntimeError(
            f"Expected {config['n_hvg']} global HVGs, "
            f"but selected {n_selected}."
        )

    selected_indices = (
        np.flatnonzero(hvg_mask)
        .astype(np.int64)
    )

    selected_genes = (
        feature_names[
            selected_indices
        ]
        .astype(str)
        .to_numpy()
    )

    library_sizes = np.asarray(
        counts.sum(axis=1)
    ).ravel()

    zero_library_cells = int(
        np.sum(library_sizes <= 0)
    )

    zero_hvg_cells = int(
        np.sum(
            np.asarray(
                counts[
                    :,
                    selected_indices,
                ].sum(axis=1)
            ).ravel()
            <= 0
        )
    )

    batch_sizes = (
        method_adata.obs[
            batch_key
        ]
        .astype(str)
        .value_counts()
        .sort_values()
    )

    if int(
        batch_sizes.min()
    ) <= int(
        config["integration_dims"]
    ):
        raise RuntimeError(
            "The fixed Seurat recipe uses "
            f"{config['integration_dims']} integration dimensions, "
            "but the smallest batch contains only "
            f"{int(batch_sizes.min())} cells."
        )

    print(
        "Dataset 2 registered feature universe:",
        f"{method_adata.n_vars:,}",
    )
    print(
        "Dataset 2 global HVGs:",
        f"{n_selected:,}",
    )
    print(
        "Zero-library cells:",
        f"{zero_library_cells:,}",
        "(retained with zero-safe LogNormalize)",
    )
    print(
        "Zero-count cells across fixed HVGs:",
        f"{zero_hvg_cells:,}",
    )
    print(
        "Integration batches:",
        len(batch_sizes),
        "min=",
        int(batch_sizes.min()),
        "max=",
        int(batch_sizes.max()),
    )

    io.mmwrite(
        workdir / "counts.mtx",
        counts.T.tocoo(),
    )

    pd.DataFrame({
        "gene": feature_names.to_numpy(),
    }).to_csv(
        workdir / "genes.csv",
        index=False,
    )

    original_barcodes = (
        method_adata.obs_names
        .astype(str)
        .to_numpy(copy=True)
    )

    pd.DataFrame({
        "barcode": original_barcodes,
    }).to_csv(
        workdir / "barcodes.csv",
        index=False,
    )

    # Only method-required batch metadata; no biological labels.
    pd.DataFrame({
        "barcode": original_barcodes,
        "BATCH": (
            method_adata.obs[
                batch_key
            ]
            .astype(str)
            .to_numpy()
        ),
    }).to_csv(
        workdir / "metadata.csv",
        index=False,
    )

    pd.DataFrame({
        "index_1based": (
            selected_indices + 1
        ),
    }).to_csv(
        workdir / "hvg_indices.csv",
        index=False,
    )

    hvg_path = (
        workdir
        / "dataset2_global_hvgs.txt"
    )
    hvg_path.write_text(
        "\n".join(
            selected_genes.tolist()
        ),
        encoding="utf-8",
    )

    policy_path = (
        workdir
        / "dataset2_method_policy.json"
    )
    policy_path.write_text(
        json.dumps(
            {
                "dataset": "mbdrc_renal_cortex",
                "feature_universe_policy": (
                    "all_registered_dataset2_features"
                ),
                "feature_universe_n": int(
                    method_adata.n_vars
                ),
                "hvg_policy": config["hvg_policy"],
                "n_hvg": n_selected,
                "hvg_span": float(
                    config["hvg_span"]
                ),
                "hvg_batch_key": None,
                "integration_batch_key": (
                    batch_key
                ),
                "normalize_method": "Seurat_LogNormalize",
                "normalize_scale_factor": float(
                    config[
                        "normalize_scale_factor"
                    ]
                ),
                "npcs": int(
                    config["npcs"]
                ),
                "integration_dims": int(
                    config[
                        "integration_dims"
                    ]
                ),
                "zero_library_cells": (
                    zero_library_cells
                ),
                "zero_hvg_cells": (
                    zero_hvg_cells
                ),
                "batch_min_cells": int(
                    batch_sizes.min()
                ),
                "batch_max_cells": int(
                    batch_sizes.max()
                ),
                "reference_policy": str(
                    config[
                        "reference_policy"
                    ]
                ),
                "reference_batch": str(
                    config[
                        "reference_batch"
                    ]
                ),
                "reference_donor": str(
                    config[
                        "reference_donor"
                    ]
                ),
                "reference_assay": str(
                    config[
                        "reference_assay"
                    ]
                ),
                "reference_n_cells": int(
                    config[
                        "reference_n_cells"
                    ]
                ),
                "k_weight": int(
                    config[
                        "k_weight"
                    ]
                ),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    del hvg_adata

    return {
        "barcodes": original_barcodes,
        "hvg_path": hvg_path,
        "policy_path": policy_path,
    }


def read_latent(
    path,
    expected_barcodes,
    expected_dims,
):
    frame = pd.read_csv(path)

    if "barcode" not in frame.columns:
        raise RuntimeError(
            f"Missing barcode column in {path}."
        )

    barcodes = (
        frame.pop("barcode")
        .astype(str)
        .to_numpy()
    )
    expected = np.asarray(
        expected_barcodes,
        dtype=str,
    )

    if (
        len(np.unique(barcodes))
        != len(barcodes)
    ):
        raise RuntimeError(
            f"Duplicate barcodes in {path.name}."
        )

    if len(barcodes) != len(expected):
        raise RuntimeError(
            f"{path.name}: {len(barcodes):,} cells; "
            f"expected {len(expected):,}."
        )

    if set(barcodes) != set(expected):
        raise RuntimeError(
            f"{path.name} cell set differs from Dataset 2."
        )

    frame.index = barcodes
    frame = frame.loc[expected]

    latent = frame.to_numpy(
        dtype=np.float32
    )

    expected_shape = (
        len(expected),
        int(expected_dims),
    )

    if latent.shape != expected_shape:
        raise RuntimeError(
            f"{path.name} latent shape {latent.shape}; "
            f"expected {expected_shape}."
        )

    if not np.isfinite(latent).all():
        raise ValueError(
            f"{path.name} contains NaN/Inf."
        )

    return latent, expected.copy()


def run_r_streaming(
    command,
    log_path,
):
    lines = []

    with Path(log_path).open(
        "w",
        encoding="utf-8",
    ) as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        if process.stdout is None:
            raise RuntimeError(
                "Could not capture R process output."
            )

        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
            log_handle.flush()
            lines.append(line)

        return_code = process.wait()

    if return_code != 0:
        tail = "".join(lines[-120:])
        raise RuntimeError(
            "Seurat R process failed with exit code "
            f"{return_code}.\n"
            f"Full R log: {log_path}\n\n"
            "---- Last R log lines ----\n"
            + tail
        )


def run_seurat_pair(
    method_adata,
    seed,
    config,
):
    seed = int(seed)

    cache_key = (
        seed,
        int(config["n_hvg"]),
        float(config["hvg_span"]),
        float(
            config[
                "normalize_scale_factor"
            ]
        ),
        int(config["npcs"]),
        int(
            config[
                "integration_dims"
            ]
        ),
        str(
            config[
                "reference_batch"
            ]
        ),
        int(
            config[
                "k_weight"
            ]
        ),
    )

    if cache_key in SEURAT_CACHE:
        print(
            f"Using cached Seurat outputs for seed {seed}."
        )
        return SEURAT_CACHE[
            cache_key
        ]

    workdir = (
        SEURAT_ROOT
        / f"seed_{seed}"
    )

    if workdir.exists():
        shutil.rmtree(workdir)

    workdir.mkdir(
        parents=True,
        exist_ok=True,
    )

    prepared = prepare_seurat_inputs(
        method_adata,
        workdir,
        config,
    )

    script_path = (
        workdir
        / "run_seurat_dataset2.R"
    )
    script_path.write_text(
        R_SCRIPT,
        encoding="utf-8",
    )

    r_log_path = (
        workdir
        / "seurat_run.log"
    )

    command = [
        "Rscript",
        str(script_path),
        str(workdir),
        str(seed),
        str(
            int(
                config["npcs"]
            )
        ),
        str(
            int(
                config[
                    "integration_dims"
                ]
            )
        ),
        str(
            float(
                config[
                    "normalize_scale_factor"
                ]
            )
        ),
        str(
            config[
                "reference_batch"
            ]
        ),
        str(
            int(
                config[
                    "k_weight"
                ]
            )
        ),
    ]

    print(
        f"\nRunning Seurat RPCA + CCA for seed {seed} ..."
    )

    run_r_streaming(
        command,
        r_log_path,
    )

    expected = prepared[
        "barcodes"
    ]
    dims = int(
        config[
            "integration_dims"
        ]
    )

    rpca = read_latent(
        workdir
        / "seurat_rpca_latent.csv",
        expected,
        dims,
    )

    cca = read_latent(
        workdir
        / "seurat_cca_latent.csv",
        expected,
        dims,
    )

    policy_payload = json.loads(
        prepared[
            "policy_path"
        ].read_text(
            encoding="utf-8"
        )
    )
    policy_payload[
        "seed"
    ] = seed

    prepared[
        "policy_path"
    ].write_text(
        json.dumps(
            policy_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    provenance = {
        "seurat_runner": script_path,
        "seurat_run_log": r_log_path,
        "seurat_session_info": (
            workdir
            / "seurat_session_info.txt"
        ),
        "seurat_layers": (
            workdir
            / "seurat_layers.txt"
        ),
        "seurat_reference": (
            workdir
            / "seurat_reference.txt"
        ),
        "dataset2_global_hvgs": (
            prepared[
                "hvg_path"
            ]
        ),
        "dataset2_method_policy": (
            prepared[
                "policy_path"
            ]
        ),
    }

    for name, path in (
        provenance.items()
    ):
        if not Path(path).exists():
            raise FileNotFoundError(
                f"Missing provenance file {name}: {path}"
            )

    SEURAT_CACHE[
        cache_key
    ] = {
        "rpca": (
            *rpca,
            provenance,
        ),
        "cca": (
            *cca,
            provenance,
        ),
    }

    print(
        f"Seed {seed} complete: "
        f"RPCA={rpca[0].shape}, "
        f"CCA={cca[0].shape}"
    )

    return SEURAT_CACHE[
        cache_key
    ]


def run_seurat_rpca(
    method_adata,
    seed,
    config,
):
    latent, barcodes, provenance = (
        run_seurat_pair(
            method_adata,
            seed,
            config,
        )["rpca"]
    )

    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key=(
            "X_seurat_rpca"
        ),
        provenance_files=provenance,
    )


def run_seurat_cca(
    method_adata,
    seed,
    config,
):
    latent, barcodes, provenance = (
        run_seurat_pair(
            method_adata,
            seed,
            config,
        )["cca"]
    )

    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key=(
            "X_seurat_cca"
        ),
        provenance_files=provenance,
    )


print(
    "Reference-aligned Dataset 2 Seurat runner loaded."
)


## Full seed-42 reference-based Seurat preflight

This is a real full-data run using the fixed single reference selected above.
Both RPCA and CCA must complete and return all 97,125 benchmark cells before
the multi-seed benchmark starts. Seed 42 is cached and reused.


In [ ]:
PREFLIGHT_SEED = int(METHOD_SEEDS[0])

preflight = run_seurat_pair(
    adata,
    PREFLIGHT_SEED,
    METHOD_CONFIG,
)

expected_n = adata.n_obs
expected_d = int(
    METHOD_CONFIG[
        "integration_dims"
    ]
)

for name in ("rpca", "cca"):
    latent, barcodes, provenance = (
        preflight[name]
    )

    if latent.shape != (
        expected_n,
        expected_d,
    ):
        raise RuntimeError(
            f"Preflight {name} shape mismatch: {latent.shape}"
        )

    if not np.array_equal(
        barcodes,
        adata.obs_names.astype(str).to_numpy(),
    ):
        raise RuntimeError(
            f"Preflight {name} barcode order mismatch."
        )

    if not np.isfinite(latent).all():
        raise RuntimeError(
            f"Preflight {name} contains NaN/Inf."
        )

    print(
        f"Preflight {name.upper()}: "
        f"{latent.shape}, finite=True, aligned=True"
    )

print(
    "\nFULL SEURAT PREFLIGHT PASSED for seed",
    PREFLIGHT_SEED,
)


## 7. Run the High-Level multi-seed benchmarks

Both methods use seeds `42, 123, 2026`. Seed 42 is already cached from the
preflight, so only seeds 123 and 2026 require new Seurat executions.


In [ ]:
from pathlib import Path

from scrarebench import (
    MethodSpec,
    benchmark_method,
)

methods = [
    MethodSpec(
        name="Seurat-RPCA",
        runner=run_seurat_rpca,
        config=METHOD_CONFIG,
    ),
    MethodSpec(
        name="Seurat-CCA",
        runner=run_seurat_cca,
        config=METHOD_CONFIG,
    ),
]

BENCHMARK_CONFIG = {
    "random_state": BENCHMARK_SEED,
    "run_scib": True,
    "scib_n_hvg": 4000,
    "scib_reference_n_pcs": 50,
        "scib_hvg_batch_mode": "global",
    "scib_n_jobs": 1,
    "scib_progress_bar": True,
    "scib_include_silhouette_batch": True,
    "scib_require_success": True,
    "include_latent_in_bundle": False,
}

results = {}

for method in methods:
    print(
        f"\n===== Benchmarking {method.name} ====="
    )

    output_dir = (
        Path(OUTPUT_ROOT)
        / method.name
    )

    result = benchmark_method(
        adata,
        method,
        seeds=METHOD_SEEDS,
        output_dir=output_dir,
        benchmark_config=(
            BENCHMARK_CONFIG
        ),
        install_dependencies=False,
        finalize=True,
        copy_input_per_seed=False,
        overwrite=True,
    )

    if not result.runs:
        raise RuntimeError(
            f"{method.name} produced no benchmark runs."
        )

    if tuple(result.seeds) != tuple(
        METHOD_SEEDS
    ):
        raise RuntimeError(
            f"{method.name} completed unexpected seeds: "
            f"{result.seeds}"
        )

    results[
        method.name
    ] = result

    print(
        f"{method.name}: completed seeds "
        f"{result.seeds}"
    )

if set(results) != {
    "Seurat-RPCA",
    "Seurat-CCA",
}:
    raise RuntimeError(
        f"Incomplete result dictionary: {list(results)}"
    )

print("\nAll Seurat benchmarks completed successfully.")


## 8. Inspect finalized summaries


In [ ]:
import pandas as pd
from IPython.display import display

if (
    "results" not in globals()
    or not results
):
    raise RuntimeError(
        "No benchmark results are available. "
        "Run the benchmark cell above first."
    )

summary_frames = []

for name, result in results.items():
    frame = result.summary().copy()
    frame.insert(
        0,
        "method",
        name,
    )
    summary_frames.append(
        frame
    )

if not summary_frames:
    raise RuntimeError(
        "Benchmark result summaries are empty."
    )

summary = pd.concat(
    summary_frames,
    ignore_index=True,
)

display(
    summary.round(5)
)

for name, result in results.items():
    print(f"\n{name}")
    print("  Seeds:", result.seeds)
    print(
        "  Multi-seed report:",
        result.report_path,
    )
    print(
        "  Delivery archive:",
        result.archive_path,
    )
    print(
        "  Summary JSON:",
        result.summary_path,
    )

    for seed, run in result.runs.items():
        print(
            f"    seed {seed}: "
            f"{run.representation_key} "
            f"-> {run.output_dir}"
        )


## 9. Download both finalized Seurat archives


In [ ]:
from pathlib import Path
from zipfile import (
    ZIP_DEFLATED,
    ZipFile,
)
from google.colab import files

if (
    "results" not in globals()
    or set(results) != {
        "Seurat-RPCA",
        "Seurat-CCA",
    }
):
    raise RuntimeError(
        "Both finalized benchmark results are required "
        "before download."
    )

download_items = {}

for method_name, result in (
    results.items()
):
    archive_path = Path(
        result.archive_path
    )

    if not archive_path.exists():
        raise FileNotFoundError(
            f"Final archive for {method_name} "
            f"does not exist: {archive_path}"
        )

    download_items[
        method_name
    ] = archive_path

    print(
        f"{method_name}: "
        f"{archive_path.name} "
        f"({archive_path.stat().st_size / (1024**2):.2f} MB)"
    )

combined_archive = Path(
    "/content/"
    "scRareBench_Seurat_Dataset2_FINAL_outputs.zip"
)

if combined_archive.exists():
    combined_archive.unlink()

with ZipFile(
    combined_archive,
    "w",
    compression=ZIP_DEFLATED,
) as zf:
    for (
        method_name,
        archive_path,
    ) in download_items.items():
        zf.write(
            archive_path,
            arcname=(
                f"{method_name}/"
                f"{archive_path.name}"
            ),
        )

if not combined_archive.exists():
    raise FileNotFoundError(
        "Combined Seurat archive was not created."
    )

print(
    f"\nDownloading: "
    f"{combined_archive.name}"
)
print(
    f"Total size: "
    f"{combined_archive.stat().st_size / (1024**2):.2f} MB"
)

files.download(
    str(combined_archive)
)
